# PERSEUS Ablation Study — CaRB (TEMPLATE, heavily redacted)

Reuses the CaRB rerun outputs. No re-extraction. Runs four ablation variants over the same Raw LLaMA triples and evaluates each against the rerun's GT.

> This template shows only cell structure and pipeline stages. Prompts, thresholds, algorithm bodies, and IO/logging details are redacted.

**Variants:** `full`, `no_repair`, `no_local_nli`, `no_context`, `no_retrieval`.

**Per variant:** feed `02_extraction_parsed_triples_dedup.csv` → modified pipeline → ACCEPT set → evaluate against same GT → McNemar vs Raw LLaMA.

## Cell 0 — Configuration

In [ ]:
import os, sys, json, re, math, time, warnings
from datetime import datetime
from pathlib import Path
from collections import Counter, defaultdict
from difflib import SequenceMatcher
import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')

# ---- Point at the completed CaRB rerun folder -----------------------
CARB_RERUN_FOLDER = '<REDACTED>'
PROJECT_ROOT      = Path(r'<REDACTED>')
RERUN_DIR         = PROJECT_ROOT / 'outputs' / CARB_RERUN_FOLDER

RUN_ID  = datetime.now().strftime('%Y%m%d_%H%M%S')
ABL_DIR = PROJECT_ROOT / 'outputs' / f'perseus_ablation_{RUN_ID}'
# <REDACTED: mkdirs, log path, Tee stdout wrapper, existence check on RERUN_DIR>

# ---- Models + thresholds (must match perseus_rerun.ipynb) -----------
HF_TOKEN       = '<REDACTED>'
NLI_MODEL_NAME = 'roberta-large-mnli'
SBERT_MODEL    = 'all-mpnet-base-v2'
# <REDACTED: R1_*, NLI_*, WEAK_*, BM25_*, TIER* thresholds>
# <REDACTED: STOPWORDS, NEGATION_CUES, MODALITY_CUES, COPULAS, PLACEHOLDERS, PRONOUNS>

# ---- Save 00_run_config.json ----------------------------------------
# <REDACTED: dump run_id, source_rerun, variants list, threshold snapshot>


## Cell 1 — Load models (NLI + SBERT only; no Llama needed)

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sentence_transformers import SentenceTransformer, util

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# <REDACTED: load nli_tokenizer, nli_model (on DEVICE), cache nli_id2label>
nli_tokenizer = None
nli_model     = None
nli_id2label  = None

# <REDACTED: load sbert_model>
sbert_model = None


## Cell 2 — Load existing CaRB rerun outputs

In [ ]:
# Read three artifacts produced by the main rerun:
#   01_carb_ground_truth_normalized.csv   -> gt_unique
#   01_carb_sentences.csv                 -> SENTENCES dict + ALL_SENT_IDS
#   02_extraction_parsed_triples_dedup.csv -> RAW_LLAMA_TRIPLES
# Sanity-check columns on RAW_LLAMA_TRIPLES.
# <REDACTED>

gt_unique         = None
SENTENCES         = None
ALL_SENT_IDS      = None
RAW_LLAMA_TRIPLES = None


## Cell 3 — Pipeline functions (shared across variants)

Mirror of the main rerun's helpers: text normalization, R1+R2 repair, NLI, lexical check, windowed premise, BM25.

In [ ]:
# ---- Text utilities -------------------------------------------------
def norm_text(s):            """Lowercase, strip non-alphanumeric, collapse whitespace."""; pass
def canon_entity(s):         """norm_text minus stopwords."""; pass
def norm_pred(p):            """Strip leading/trailing copulas from predicate."""; pass
def normalize_triples_df(df):"""Add Snorm/Pnorm/Onorm columns, drop invalid rows."""; pass

# ---- R1 + R2 repair -------------------------------------------------
PRED_NORM_MAP = {'<REDACTED>': '...'}

def best_span(query, sentence, min_score):
    """Best contiguous sentence span for query (weighted token-overlap +
    SequenceMatcher ratio)."""
    pass

def repair_one(subj, pred, obj, sent):
    """R1 (canonicalize + preserve negation/modality) + R2 (content grounding).
    Returns ((s, p, o), None, edits) on success or (None, drop_reason, edits)."""
    pass

# ---- NLI ------------------------------------------------------------
def triple_to_hyp(s, p, o):  """Build NLI hypothesis string from triple."""; pass
def nli_s(premise, hyp):     """Run NLI, return {'entail', 'neutral', 'contra'}."""; pass
def decide_nli(sc):          """ACCEPT / REJECT / UNVERIFIED via thresholds."""; pass
def lex_score(s, o, sent):   """0/1/2 count of (subj, obj) appearing in sentence."""; pass
def get_window(sent_id, radius=1):
    """Concatenate +/- radius neighboring CaRB sentences."""
    pass

# ---- BM25 -----------------------------------------------------------
def tokenize_bm25(s):              """BM25 tokenizer."""; pass
def bm25_score(qtoks, doc_idx):    """Okapi BM25 score."""; pass
def retrieve_topk(q, k, exclude):  """Top-k ALL_SENT_IDS by BM25."""; pass

# <REDACTED: build corpus stats (doc_tokens, doc_lens, avgdl, df counts) for BM25>


## Cell 4 — Variant runner

Each variant is parameterized by four flags: `use_repair`, `use_local_nli`, `use_window`, `use_retrieval`. Runs repair → validation per those flags, saves per-variant ACCEPT/REJECT CSVs.

In [ ]:
def run_variant(variant_name, use_repair, use_local_nli, use_window, use_retrieval):
    """Run one ablation variant end-to-end.

    Flow:
      1. Repair stage (or forward raw triples if use_repair=False).
      2. For each triple:
          - lex_score; if not use_local_nli, accept when lex >= 1, else reject.
          - If lex == 0: immediate reject.
          - Local NLI -> decide_nli.
          - If UNVERIFIED and use_window: escalate to windowed NLI.
          - If use_retrieval: trigger on weak-ACCEPT or UNVERIFIED,
            run NLI over BM25 top-k with early exit on ACCEPT/REJECT.
      3. Save 05_<variant>_ACCEPT.csv and 05_<variant>_REJECT.csv.

    Returns (accept_df, nli_calls_count).
    """
    pass

variant_configs = [
    # (name,           use_repair, use_local_nli, use_window, use_retrieval)
    ('full',           True,       True,          True,       True),
    ('no_repair',      False,      True,          True,       True),
    ('no_local_nli',   True,       False,         False,      False),
    ('no_context',     True,       True,          False,      True),
    ('no_retrieval',   True,       True,          True,       False),
]

VARIANT_PREDS     = {}
VARIANT_NLI_CALLS = {}

# <REDACTED: loop calling run_variant for each config, populate dicts>


## Cell 5 — Evaluate each variant against CaRB GT

Same 3-tier SBERT matcher as the main rerun. Also evaluates Raw LLaMA baseline for McNemar pairs.

In [ ]:
def evaluate_preds(pred_df, gt_unique_df, variant_name):
    """3-tier matching (exact -> SBERT tier2 -> SBERT tier3, arg-swap tolerant,
    greedy 1-1 assignment). Returns (metrics_dict, gt_used_set)."""
    pass

# Baseline first (for McNemar pairs)
baseline_metrics, baseline_gt_matched = evaluate_preds(RAW_LLAMA_TRIPLES, gt_unique, 'baseline')

VARIANT_METRICS    = {'baseline': baseline_metrics}
VARIANT_GT_MATCHED = {'baseline': baseline_gt_matched}

# <REDACTED: loop over ['full', 'no_repair', 'no_local_nli', 'no_context', 'no_retrieval']
#  calling evaluate_preds, populating dicts, printing P/R/F1 per variant>


## Cell 6 — McNemar tests (each variant vs Raw LLaMA baseline) + summary

In [ ]:
from statsmodels.stats.contingency_tables import mcnemar as mcnemar_test

TOTAL_GT    = len(gt_unique)
baseline_gt = VARIANT_GT_MATCHED['baseline']

# For each variant vs baseline:
#   Contingency: a=both matched, b=variant only, c=baseline only, d=neither.
#   Exact binomial if (b+c) < 25 else chi-squared Yates-corrected.
#   Record p-value, discordant counts, delta F1 vs full pipeline.
# <REDACTED: comparison loop>

summary_rows = []   # rows for 06_ablation_summary.csv / json
# <REDACTED: build rows, save:
#  - 06_ablation_summary.csv
#  - 06_ablation_summary.json
#  - 10_FINAL_SUMMARY.txt (formatted table with P/R/F1, delta vs Full, McNemar p)>

# <REDACTED: close log file, restore stdout, print completion banner>
